# Markov Chain–Based Behavioral Predictability Analysis

To assess how predictable each participant’s behavior was during the Rock–Paper–Scissors game, the authors applied a Markov chain analysis to the behavioral response data. The goal of this analysis was not to model an "optimal play", but to quantify the extent to which a participant’s next response could be predicted from their previous responses. The Prediction accuracy was used as a direct measure of behavioral predictability.

The analysis was conducted separately for each participant. For every trial in the experiment, the authors tracked the participant’s responses across time and computed how often each response (Rock, Paper, or Scissors) followed a specific response on the previous trial.
The resulting prediction accuracy from the stochastical markov analysis allowed the authors of the study to consee whether participants behaved randomly or followed some quasi systematic patterns during the trails.

## Behavioral Data Structure

The behavioral responses used for the Markov analysis are stored in
**TSV** files for each participant pair. Each row
corresponds to a single trial in the experiment and contains timing
information, the responses of both players, their reaction times, and
the outcome of the round.

A simplified example from the original data structure is shown below:

| onset | duration | onset_sample | trial_num | player1_resp | player1_rt | player2_resp | player2_rt | outcome |
|------|--------|-------------|---------|-------------|-----------|-------------|-----------|-------|
| 420.78 | 5 | 861754 | 1 | 2 | 0.025 | 1 | 1.308 | 2 |
| 425.77 | 5 | 871977 | 2 | 3 | 1.51 | 1 | 1.11 | 3 |

The columns later relevant for the conducted Markov analysis are the response columns:

- **player1_resp** – response of player 1  
- **player2_resp** – response of player 2  

Responses are encoded numerically:

- **1 = Rock**
- **2 = Paper**
- **3 = Scissors**
- **0 = No response**

For each pair of participants, a nuimber of  **480 trials** was recorded and extracted from the **TSV** files in order to be processed as an input for the Markov chain algorithm..



---

## First-Order Markov Chain Model

The behavioral predictability analysis is based on a **first-order Markov chain**, a probabilistic model commonly used to describe sequential processes in which the probability of the next state depends only on the current state.

Formally, the **Markov property** states that the future probability in a markov chain of random variable is conditionally independent of all earlier states(prababilities) given the present state. In mathematical terms:

$$
P(X_{t+1} \mid X_t, X_{t-1}, ..., X_1) = P(X_{t+1} \mid X_t)
$$

where  
- \(X_t\) denotes the response at trial \(t\)  
- \(X_{t+1}\) denotes the response at the next trial  

This property implies that the model **does not consider** the entire history of responses, but only the most recent one when predicting the next action.




# Transition Counts

The first step of the implementation constructs cumulative counts of
response transitions across trials. These counts are stored in a matrix
called `prob_data` with dimensions:

$$
(\text{num_trials} \times 13)
$$

Each row corresponds to a trial and contains cumulative counts of
transitions observed up to that trial.

  Column   Meaning
  -------- --------------------------------------------------------------
  0        Trial index
  1        Number of Rock responses
  2--4     Rock→Rock, Rock→Paper, Rock→Scissors transitions
  5        Number of Paper responses
  6--8     Paper→Rock, Paper→Paper, Paper→Scissors transitions
  9        Number of Scissors responses
  10--12   Scissors→Rock, Scissors→Paper, Scissors→Scissors transitions

During each iteration, the counts from the previous trial are copied and
updated according to the current transition.

``` python
prob_data[i, :] = prob_data[i - 1, :]
```

If the previous response was Rock, Paper, or Scissors, the corresponding
transition counts are incremented.

------------------------------------------------------------------------

# Sliding Window Transition Estimation

Rather than computing transition probabilities from the entire response
history, the algorithm estimates them using a **sliding window** of
recent trials.

Window sizes between **5 and 100 trials** are tested. For each trial
$i$, the transition counts inside the window are computed by subtracting
cumulative counts:

$$
\text{window transitions} =
\text{prob_data}[i-1] - \text{prob_data}[i-\text{window_size}]
$$

If the current trial occurs early in the experiment and a full window is
not yet available, all transitions observed up to that trial are used
instead.




### Transition Probabilities

The behavior of the Markov chain is fully defined by its **transition probabilities**, which describe the likelihood of moving from one response to another between consecutive trials.

In our code we have for example:

- \(P(P|R)\) — probability of playing **Paper after Rock**
- \(P(S|P)\) — probability of playing **Scissors after Paper**
- \(P(R|S)\) — probability of playing **Rock after Scissors**

These probabilities are given empirically from the observed responses in the studied trails.

For instance, the conditional probability of Paper following Rock is calculated as:

$$
P(P|R) =
\frac{\text{Number of transitions } R \rightarrow P}
{\text{Total number of transitions starting from } R}
$$

This estimation approach corresponds to the **maximum likelihood estimate (MLE)** of the transition probabilities given the observed data.



## Transition Matrix

The core of the Markov model and also in the paper implementation is the **transition probability matrix**.
This matrix contains the probabilities of moving from one response to
another and has size (3 `\times 3`{=tex}).

$$
T =
\begin{bmatrix}
P(R|R) & P(P|R) & P(S|R) \\
P(R|P) & P(P|P) & P(S|P) \\
P(R|S) & P(P|S) & P(S|S)
\end{bmatrix}
$$

Each **row corresponds to the previous response**, while each **column
corresponds to the predicted next response**.

Example interpretation:

  Previous Response   Rock      Paper     Scissors
  ------------------- --------- --------- ----------
  Rock                P(R\|R)   P(P\|R)   P(S\|R)
  Paper               P(R\|P)   P(P\|P)   P(S\|P)
  Scissors            P(R\|S)   P(P\|S)   P(S\|S)

If a participant frequently plays **Paper after Rock**, then (P(P\|R))
will be high. If the participant behaves randomly, all probabilities
approach **1/3**.

Transition probabilities are computed as:

$$
P(P|R) = \frac{\text{Number of times Paper followed Rock}}{\text{Total number of Rock responses}}
$$

## Sliding Window Estimation

Instead of computing probabilities from the entire experiment, the
analysis uses a **sliding window over the previous (N) trials**. Window
sizes between **5 and 100 trials** are evaluated.

For trial (i), the transition probabilities are estimated using
transitions within the window:

$$
T_i = \text{Transitions observed in trials } [i-N, ..., i-1]
$$

If a particular response does not occur in the window, the probabilities
default to **1/3**, representing uncertainty.

## Prediction Using Argmax

Once the transition matrix is computed, the model predicts the next
response based on the **most recent valid response**.

If the previous response was (X_t), the model retrieves the probability
vector:

$$
[P(R|X_t), P(P|X_t), P(S|X_t)]
$$

The predicted response is the one with the **highest probability**,
which is obtained using the **argmax** operation.

Example probability vector:


